In [1]:
import os
import ee
import geemap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import geopandas as gpd

import plotly.express as px
import plotly.graph_objects as go


import random
from shapely.geometry import Polygon, mapping

In [3]:
m = geemap.Map(center=[40,-100], zoom=4)
m

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [51]:
df_big = pd.read_csv('Big_creek/big_data/big_model_database.csv')
df_brushy = pd.read_csv('Brushy_Creek/brushy_data/brushy_model_database.csv')
df_hogpen = pd.read_csv('Hogpen_creek/hogpen_data/hogpen_model_database.csv')
df_big.head()

,Date,Site,Turbidity,Solar,max_rain_event,mean_rain_event,rain_event_date,drainage_area_km2,Soil (HSG D),Soil (HSG C),Soil (HSG B),area_bare land,area_built-up,area_grassland,area_solar panels,area_tree vegetation,area_water
0,1/22/2024,Big@162_downstream,83.70,1,0.0,0.0,NaN,82.357625,81.67,0.55,0.23,35.380320,4.973129,24.678866,10.008276,6.488527,0.806392
1,1/22/2024,Big@168_ref,1021.00,0,0.0,0.0,NaN,57.929038,56.96,0.00,0.23,26.972473,4.085662,22.205585,0.007187,3.985287,0.656112
2,1/23/2024,Big@162_downstream,2712.00,1,0.0,0.0,NaN,82.357625,81.67,0.55,0.23,35.380320,4.973129,24.678866,10.008276,6.488527,0.806392
3,1/23/2024,Big@168_ref,384.50,0,0.0,0.0,NaN,57.929038,56.96,0.00,0.23,26.972473,4.085662,22.205585,0.007187,3.985287,0.656112
4,1/24/2024,Big@162_downstream,57.05,1,0.0,0.0,NaN,82.357625,81.67,0.55,0.23,35.380320,4.973129,24.678866,10.008276,6.488527,0.806392


In [52]:
print(df_big.columns.tolist())
print('"""" '*20)
print(df_brushy.columns.tolist())
print('"""" '*20)
print(df_hogpen.columns.tolist())

['Date', 'Site', 'Turbidity', 'Solar', 'max_rain_event', 'mean_rain_event', 'rain_event_date', 'drainage_area_km2', 'Soil (HSG D)', 'Soil (HSG C)', 'Soil (HSG B)', 'area_bare land', 'area_built-up', 'area_grassland', 'area_solar panels', 'area_tree vegetation', 'area_water']
"""" """" """" """" """" """" """" """" """" """" """" """" """" """" """" """" """" """" """" """" 
['Date', 'Site', 'Turbidity', 'Solar', 'max_rain_event', 'mean_rain_event', 'rain_event_date', 'drainage_area_km2', 'Soil (HSG D)', 'Soil (HSG C)', 'Soil (HSG B)', 'area_bare land', 'area_built-up', 'area_grassland', 'area_solar panels', 'area_tree vegetation', 'area_water']
"""" """" """" """" """" """" """" """" """" """" """" """" """" """" """" """" """" """" """" """" 
['Date', 'Site', 'Turbidity', 'Solar', 'max_rain_event', 'mean_rain_event', 'rain_event_date', 'drainage_area_km2', 'Soil (HSG D)', 'Soil (HSG C)', 'Soil (HSG B)', 'area_bare land', 'area_built-up', 'area_grassland', 'area_solar panels', 'area_tr

In [53]:
combined_df = pd.concat([df_big, df_brushy, df_hogpen], ignore_index=True)
#combined_df.tail(50)
combined_df.iloc[120:]

,Date,Site,Turbidity,Solar,max_rain_event,mean_rain_event,rain_event_date,drainage_area_km2,Soil (HSG D),Soil (HSG C),Soil (HSG B),area_bare land,area_built-up,area_grassland,area_solar panels,area_tree vegetation,area_water
120,4/10/2024,Hogpen@LCR 608_ref,209.000000,0,5.08,5.065810,4/10/2024,19.734360,19.73,0.00,0.00,12.451437,0.440181,4.508804,0.000000,0.427583,0.160132
121,4/29/2024,Hogpen@162_downstream,229.000000,1,1.76,1.170056,4/29/2024,35.311554,35.05,0.16,0.02,24.894394,0.823350,7.202780,0.343036,1.931837,0.378163
122,4/29/2024,Hogpen@174_impact,170.666667,1,1.76,1.170056,4/29/2024,31.091089,30.87,0.16,0.00,23.002368,0.655547,6.056663,0.000000,1.442478,0.343197
123,4/29/2024,Hogpen@LCR 608_ref,106.333333,0,1.76,1.170056,4/29/2024,19.734360,19.73,0.00,0.00,12.451437,0.440181,4.508804,0.000000,0.427583,0.160132
124,5/31/2024,Hogpen@162_downstream,210.666667,1,3.60,3.600000,5/31/2024,35.311554,35.05,0.16,0.02,24.894394,0.823350,7.202780,0.343036,1.931837,0.378163
125,5/31/2024,Hogpen@174_impact,57.266667,1,3.60,3.600000,5/31/2024,31.091089,30.87,0.16,0.00,23.002368,0.655547,6.056663,0.000000,1.442478,0.343197
126,5/31/2024,Hogpen@LCR 608_ref,148.333333,0,3.60,3.600000,5/31/2024,19.734360,19.73,0.00,0.00,12.451437,0.440181,4.508804,0.000000,0.427583,0.160132
127,1/31/2025,Hogpen@162_downstream,355.000000,1,2.38,1.086341,1/31/2025,35.311554,35.05,0.16,0.02,24.894394,0.823350,7.202780,0.343036,1.931837,0.378163
128,1/31/2025,Hogpen@174_impact,236.333333,1,2.38,1.086341,1/31/2025,31.091089,30.87,0.16,0.00,23.002368,0.655547,6.056663,0.000000,1.442478,0.343197
129,1/31/2025,Hogpen@LCR 608_ref,180.666667,0,2.38,1.086341,1/31/2025,19.734360,19.73,0.00,0.00,12.451437,0.440181,4.508804,0.000000,0.427583,0.160132


In [54]:
# Load hogpen shapefile into a GeoDataFrame
hogpen162 = geemap.shp_to_ee("Hogpen_creek/hugpen_downstream.shp")
hogpen174 = geemap.shp_to_ee("Hogpen_creek/hugpen_impact.shp")
hogpen608 = geemap.shp_to_ee("Hogpen_creek/hugpen_upstream.shp")

# Load brushy shapefile into a GeoDataFrame
brushy152 = geemap.shp_to_ee("Brushy_Creek/brushy_downstream.shp")
brushy1240 = geemap.shp_to_ee("Brushy_Creek/brushy_impact.shp")
brushy118 = geemap.shp_to_ee("Brushy_Creek/brushy_ref.shp")
Librushy150 = geemap.shp_to_ee("Brushy_Creek/Littlebrushy_ref.shp")
Librushy118 = geemap.shp_to_ee("Brushy_Creek/little_brushy_ref2.shp")

# Load big shapefile into a GeoDataFrame
big_fm162 = geemap.shp_to_ee("Big_creek/big_creek_downstream.shp")
big_fm165 = geemap.shp_to_ee("Big_creek/big_creek_impact.shp")
big_fm168 = geemap.shp_to_ee("Big_creek/big_creek_ref.shp")
big_blake = geemap.shp_to_ee("Big_creek/big_creek_upstream1.shp")
big_Lfelton = geemap.shp_to_ee("Big_creek/big_creek_upstream.shp")

In [131]:
## *
# Function to mask clouds using the Sentinel-2 QA band
# @param {ee.Image} image Sentinel-2 image
# @return {ee.Image} cloud masked Sentinel-2 image
#
map = geemap.Map(center=[40,-100], zoom=4)

roi = ee.Geometry.BBox(-97.528381, 31.781736, -96.630249, 31.292487)
aoi = ee.Geometry.Rectangle([-97.021749, 31.653679, -96.581087, 31.365537])

def maskS2clouds(image):
    qa = image.select('QA60')

    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11

    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
    .And(qa.bitwiseAnd(cirrusBitMask).eq(0))

    return image.updateMask(mask).divide(10000)
    
# .filterDate('2024-09-01', '2024-10-01') \3/28/2025

S2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
.filterDate('2024-03-26', '2024-04-24') \
.filterBounds(aoi) \
.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',10)) \
.map(maskS2clouds)

visualization = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'],
}


style1 = {'color': 'ffffff', 'fillColor': '00000000'}

map.addLayer(S2.median(), visualization, 'RGB')
map.addLayer(roi, {}, 'roi', shown=False)
map.addLayer(big_fm162, {}, 'bigfm162', shown=True)
map.centerObject(aoi)
map

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [132]:
S2.size()

In [133]:
# SS2 = S2.median()
# nir = SS2.select('B8')
# red = SS2.select('B4')
# ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

def ndvi_img(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('ndvi')
    return ndvi \
    .copyProperties(image, ['system:time_start', 'system:time_end'])


ndvi = S2.map(ndvi_img)


Map = geemap.Map()


ndviparam = {
    'min': -1,
    'max': 1,
    'palette': ['blue', 'white', 'green']
}

median_ndvi = ndvi.median()
brushy152
hogpen162
Map.addLayer(median_ndvi, ndviparam, 'ndvi')
Map.addLayer(big_fm162, {}, 'bigfm162', shown=True)
Map.addLayer(brushy152, {}, 'brushy152', shown=True)
Map.addLayer(hogpen162, {}, 'hogpen162', shown=True)
#Map.addLayer(fishnet.style(**style), {}, 'Fishnet')
Map.centerObject(aoi)
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [134]:
combined_df['Site'].unique()

array(['Big@162_downstream', 'Big@168_ref', 'Big@165_impact',
       'Big@BattleLake_upstream', 'Big@LakeFelton_upstream1',
       'Brushy@118_ref', 'Brushy@152_downstream', 'LiBrushy@118_ref1',
       'LiBrushy@150_impact1', 'Brushy@1240_impact',
       'Hogpen@162_downstream', 'Hogpen@174_impact', 'Hogpen@LCR 608_ref'],
      dtype=object)

In [135]:
site_polygons = {
    'Hogpen@162_downstream': hogpen162,
    'Hogpen@174_impact': hogpen174,
    'Hogpen@LCR 608_ref': hogpen608,
    'Brushy@152_downstream': brushy152,
    'Brushy@1240_impact': brushy1240,
    'Brushy@118_ref': brushy118,
    'LiBrushy@150_impact1': Librushy150,
    'LiBrushy@118_ref1': Librushy118,
    'Big@LakeFelton_upstream1': big_Lfelton,
    'Big@BattleLake_upstream': big_blake,
    'Big@168_ref': big_fm168,
    'Big@165_impact': big_fm165,
    'Big@162_downstream': big_fm162
}

In [136]:
# Example: checking 'Site' column values against dictionary keys
missing_keys = set(combined_df['Site'].unique()) - set(site_polygons.keys())

# Print missing keys (if any)
print("Missing keys:", missing_keys)

Missing keys: set()


In [137]:
all_keys_present = set(combined_df['Site'].unique()).issubset(site_polygons.keys())
print(all_keys_present)  # True if all are found in the dict keys

True


In [138]:
combined_df['Date'].unique()

array(['1/23/2024', '1/24/2024', '2/10/2024', '2/12/2024', '2/15/2024',
       '3/1/2024', '3/22/2024', '4/10/2024', '4/29/2024', '5/31/2024',
       '1/28/2025', '1/31/2025', '3/28/2025', '4/25/2025', '4/9/2024'],
      dtype=object)

In [139]:
combined_df['Date'].value_counts()

Date
2/10/2024    13
3/1/2024     13
2/12/2024    13
1/31/2025    13
5/31/2024    13
4/29/2024    13
4/10/2024    13
4/25/2025    11
3/22/2024     8
1/24/2024     6
1/23/2024     4
2/15/2024     4
3/28/2025     3
1/28/2025     2
4/9/2024      2
Name: count, dtype: int64

In [140]:
combined_df = combined_df[combined_df['Date'] != '1/22/2024']

In [141]:
combined_df['Date'].value_counts()

Date
2/10/2024    13
3/1/2024     13
2/12/2024    13
1/31/2025    13
5/31/2024    13
4/29/2024    13
4/10/2024    13
4/25/2025    11
3/22/2024     8
1/24/2024     6
1/23/2024     4
2/15/2024     4
3/28/2025     3
1/28/2025     2
4/9/2024      2
Name: count, dtype: int64

In [142]:
combined_df[combined_df['Date'] == '4/9/2024']

,Date,Site,Turbidity,Solar,max_rain_event,mean_rain_event,rain_event_date,drainage_area_km2,Soil (HSG D),Soil (HSG C),Soil (HSG B),area_bare land,area_built-up,area_grassland,area_solar panels,area_tree vegetation,area_water,NDVI
79,4/9/2024,Brushy@118_ref,605.333333,0,4.94,4.2396,4/9/2024,20.262951,19.36,0.74,0.12,9.048778,0.660150,8.655191,0.000000,1.632973,0.259376,0.577271
80,4/9/2024,LiBrushy@118_ref1,321.666667,0,4.94,4.2396,4/9/2024,14.179835,13.57,0.58,0.00,5.809158,0.458996,6.642117,0.171356,0.924129,0.171841,0.610014


In [151]:
#target_date = ['2/10/2024', '2/12/2024']
target_date = '4/9/2024'

In [152]:
#combined_df[combined_df['Date'].isin(target_date)]
combined_df[combined_df['Date'] == target_date]

,Date,Site,Turbidity,Solar,max_rain_event,mean_rain_event,rain_event_date,drainage_area_km2,Soil (HSG D),Soil (HSG C),Soil (HSG B),area_bare land,area_built-up,area_grassland,area_solar panels,area_tree vegetation,area_water,NDVI
79,4/9/2024,Brushy@118_ref,605.333333,0,4.94,4.2396,4/9/2024,20.262951,19.36,0.74,0.12,9.048778,0.660150,8.655191,0.000000,1.632973,0.259376,0.577271
80,4/9/2024,LiBrushy@118_ref1,321.666667,0,4.94,4.2396,4/9/2024,14.179835,13.57,0.58,0.00,5.809158,0.458996,6.642117,0.171356,0.924129,0.171841,0.610014


In [153]:
# Filter combined_df by the target date
filtered_df = combined_df[combined_df['Date'] == target_date].copy()
#filtered_df = combined_df[combined_df['Date'].isin(target_date)].copy()

# Ensure the NDVI column exists
if 'NDVI' not in combined_df.columns:
    combined_df['NDVI'] = None

# Loop over the filtered rows by site
for idx, row in filtered_df.iterrows():
    site_name = row['Site']
    if site_name in site_polygons:
        try:
            feature = site_polygons[site_name]
            
            # Reduce the NDVI raster over the site polygon (mean NDVI)
            mean_dict = median_ndvi.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=feature.geometry(),
                scale=10,
                maxPixels=1e9,    # Increased to 9 billion pixels
                bestEffort=True   # Allow Earth Engine to adjust the scale if needed
            ).getInfo()
            
            mean_ndvi = mean_dict.get('ndvi')  # or replace 'NDVI' with actual band name

            # Insert NDVI value into original DataFrame at correct row
            combined_df.at[idx, 'NDVI'] = mean_ndvi

        except Exception as e:
            print(f"Error extracting NDVI for site {site_name}: {e}")

In [154]:
filtered_df

,Date,Site,Turbidity,Solar,max_rain_event,mean_rain_event,rain_event_date,drainage_area_km2,Soil (HSG D),Soil (HSG C),Soil (HSG B),area_bare land,area_built-up,area_grassland,area_solar panels,area_tree vegetation,area_water,NDVI
79,4/9/2024,Brushy@118_ref,605.333333,0,4.94,4.2396,4/9/2024,20.262951,19.36,0.74,0.12,9.048778,0.660150,8.655191,0.000000,1.632973,0.259376,0.577271
80,4/9/2024,LiBrushy@118_ref1,321.666667,0,4.94,4.2396,4/9/2024,14.179835,13.57,0.58,0.00,5.809158,0.458996,6.642117,0.171356,0.924129,0.171841,0.610014


In [155]:
mean_dict

{'ndvi': 0.613380997421297}

In [156]:
#combined_df[combined_df['Date'].isin(target_date)]
combined_df[combined_df['Date'] == target_date]

,Date,Site,Turbidity,Solar,max_rain_event,mean_rain_event,rain_event_date,drainage_area_km2,Soil (HSG D),Soil (HSG C),Soil (HSG B),area_bare land,area_built-up,area_grassland,area_solar panels,area_tree vegetation,area_water,NDVI
79,4/9/2024,Brushy@118_ref,605.333333,0,4.94,4.2396,4/9/2024,20.262951,19.36,0.74,0.12,9.048778,0.660150,8.655191,0.000000,1.632973,0.259376,0.587603
80,4/9/2024,LiBrushy@118_ref1,321.666667,0,4.94,4.2396,4/9/2024,14.179835,13.57,0.58,0.00,5.809158,0.458996,6.642117,0.171356,0.924129,0.171841,0.613381


In [497]:
combined_df.tail(10)

,Date,Site,Turbidity,Solar,max_rain_event,mean_rain_event,rain_event_date,drainage_area_km2,Soil (HSG D),Soil (HSG C),Soil (HSG B),area_bare land,area_built-up,area_grassland,area_solar panels,area_tree vegetation,area_water,NDVI
119,4/29/2024,Hogpen@LCR 608_ref,106.333333,0,1.76,1.170056,4/29/2024,19.734360,19.73,0.00,0.00,12.451437,0.440181,4.508804,0.000000,0.427583,0.160132,0.582222
120,5/31/2024,Hogpen@162_downstream,210.666667,1,3.60,3.600000,5/31/2024,35.311554,35.05,0.16,0.02,24.894394,0.823350,7.202780,0.343036,1.931837,0.378163,0.591251
121,5/31/2024,Hogpen@174_impact,57.266667,1,3.60,3.600000,5/31/2024,31.091089,30.87,0.16,0.00,23.002368,0.655547,6.056663,0.000000,1.442478,0.343197,0.592311
122,5/31/2024,Hogpen@LCR 608_ref,148.333333,0,3.60,3.600000,5/31/2024,19.734360,19.73,0.00,0.00,12.451437,0.440181,4.508804,0.000000,0.427583,0.160132,0.541906
123,1/31/2025,Hogpen@162_downstream,355.000000,1,2.38,1.086341,1/31/2025,35.311554,35.05,0.16,0.02,24.894394,0.823350,7.202780,0.343036,1.931837,0.378163,0.395744
124,1/31/2025,Hogpen@174_impact,236.333333,1,2.38,1.086341,1/31/2025,31.091089,30.87,0.16,0.00,23.002368,0.655547,6.056663,0.000000,1.442478,0.343197,0.387826
125,1/31/2025,Hogpen@LCR 608_ref,180.666667,0,2.38,1.086341,1/31/2025,19.734360,19.73,0.00,0.00,12.451437,0.440181,4.508804,0.000000,0.427583,0.160132,0.402326
126,4/25/2025,Hogpen@162_downstream,265.000000,1,1.63,1.154583,4/25/2025,35.311554,35.05,0.16,0.02,24.894394,0.823350,7.202780,0.343036,1.931837,0.378163,0.48859
127,4/25/2025,Hogpen@174_impact,167.666667,1,1.63,1.154583,4/25/2025,31.091089,30.87,0.16,0.00,23.002368,0.655547,6.056663,0.000000,1.442478,0.343197,0.47163
128,4/25/2025,Hogpen@LCR 608_ref,133.333333,0,1.63,1.154583,4/25/2025,19.734360,19.73,0.00,0.00,12.451437,0.440181,4.508804,0.000000,0.427583,0.160132,0.466557


In [500]:
combined_df.head()

,Date,Site,Turbidity,Solar,max_rain_event,mean_rain_event,rain_event_date,drainage_area_km2,Soil (HSG D),Soil (HSG C),Soil (HSG B),area_bare land,area_built-up,area_grassland,area_solar panels,area_tree vegetation,area_water,NDVI
2,1/23/2024,Big@162_downstream,2712.000000,1,0.00,0.000000,NaN,82.357625,81.67,0.55,0.23,35.380320,4.973129,24.678866,10.008276,6.488527,0.806392,0.419691
3,1/23/2024,Big@168_ref,384.500000,0,0.00,0.000000,NaN,57.929038,56.96,0.00,0.23,26.972473,4.085662,22.205585,0.007187,3.985287,0.656112,0.437575
4,1/24/2024,Big@162_downstream,57.050000,1,0.00,0.000000,NaN,82.357625,81.67,0.55,0.23,35.380320,4.973129,24.678866,10.008276,6.488527,0.806392,0.419691
5,1/24/2024,Big@168_ref,162.000000,0,0.00,0.000000,NaN,57.929038,56.96,0.00,0.23,26.972473,4.085662,22.205585,0.007187,3.985287,0.656112,0.437575
6,2/10/2024,Big@162_downstream,31.666667,1,0.79,0.509832,2/10/2024,82.357625,81.67,0.55,0.23,35.380320,4.973129,24.678866,10.008276,6.488527,0.806392,0.404475


In [502]:
combined_df['NDVI'].isnull().any()

False

In [507]:
combined_df['NDVI'].isnull().sum()

0

In [512]:
combined_df.head(10)

,Date,Site,Turbidity,Solar,max_rain_event,mean_rain_event,rain_event_date,drainage_area_km2,Soil (HSG D),Soil (HSG C),Soil (HSG B),area_bare land,area_built-up,area_grassland,area_solar panels,area_tree vegetation,area_water,NDVI
2,1/23/2024,Big@162_downstream,2712.000000,1,0.00,0.000000,NaN,82.357625,81.67,0.55,0.23,35.380320,4.973129,24.678866,10.008276,6.488527,0.806392,0.419691
3,1/23/2024,Big@168_ref,384.500000,0,0.00,0.000000,NaN,57.929038,56.96,0.00,0.23,26.972473,4.085662,22.205585,0.007187,3.985287,0.656112,0.437575
4,1/24/2024,Big@162_downstream,57.050000,1,0.00,0.000000,NaN,82.357625,81.67,0.55,0.23,35.380320,4.973129,24.678866,10.008276,6.488527,0.806392,0.419691
5,1/24/2024,Big@168_ref,162.000000,0,0.00,0.000000,NaN,57.929038,56.96,0.00,0.23,26.972473,4.085662,22.205585,0.007187,3.985287,0.656112,0.437575
6,2/10/2024,Big@162_downstream,31.666667,1,0.79,0.509832,2/10/2024,82.357625,81.67,0.55,0.23,35.380320,4.973129,24.678866,10.008276,6.488527,0.806392,0.404475
7,2/10/2024,Big@165_impact,32.666667,1,0.79,0.509832,2/10/2024,75.364478,73.99,0.55,0.23,33.321864,4.627186,24.032686,7.064129,5.518046,0.779017,0.404432
8,2/10/2024,Big@168_ref,20.300000,0,0.79,0.509832,2/10/2024,57.929038,56.96,0.00,0.23,26.972473,4.085662,22.205585,0.007187,3.985287,0.656112,0.414603
9,2/10/2024,Big@BattleLake_upstream,24.133333,0,0.79,0.509832,2/10/2024,37.767860,36.36,0.00,0.23,17.672314,1.419867,15.716413,0.000000,2.504209,0.438404,0.411029
10,2/10/2024,Big@LakeFelton_upstream1,396.000000,0,0.79,0.509832,2/10/2024,9.963803,9.28,0.00,0.00,5.664127,0.283925,3.633368,0.000000,0.255016,0.127750,0.416021
11,2/12/2024,Big@162_downstream,256.666667,1,1.41,1.410000,2/11/2024,82.357625,81.67,0.55,0.23,35.380320,4.973129,24.678866,10.008276,6.488527,0.806392,0.404475


In [513]:
# combined_df['NDVI'] = pd.to_numeric(combined_df['NDVI'], errors='coerce')
# combined_df.to_csv('lulc/Otto_model_database.csv', index=False, float_format='%.10f')